In [1]:
from config import config
from pathlib import Path
import numpy as np
import pandas as pd
import shutil

In [2]:
METADATA_FILE = Path("../../../data/metadata/metadata_en.csv")

In [3]:
metadata_df = pd.read_csv(METADATA_FILE)
metadata_df.head()

,audio_file_stem,lyric_file_stem,quadrant,genres,popular_genre
0,MT0000008995,MT0000008995,Q3,"Blues,Jazz,Vocal",Jazz
1,MT0000011357,MT0000011357,Q2,"Electronic,International,Pop/Rock",Pop/Rock
2,MT0000026571,MT0000026571,Q4,Vocal,Vocal
3,MT0000027422,MT0000027422,Q3,Pop/Rock,Pop/Rock
4,MT0000031210,MT0000031210,Q3,"Blues,Jazz,Vocal",Jazz


In [4]:
root = Path("../../..")
FEATURES_DIR = root / config.FEATURES_DIR

In [5]:
all_feature_files = list(FEATURES_DIR.glob("*.npy"))
parent_ids = [f.stem.split('_')[0] for f in all_feature_files]
unique_song_ids = sorted(list(set(parent_ids)))

In [6]:
unique_song_ids[:5]

['A002', 'A004', 'A010', 'A013', 'A014']

In [7]:
relevant_meta = metadata_df[metadata_df['audio_file_stem'].isin(unique_song_ids)]

In [8]:
metadata_df.shape, relevant_meta.shape

((1531, 5), (1531, 5))

In [9]:
relevant_meta.columns

Index(['audio_file_stem', 'lyric_file_stem', 'quadrant', 'genres',
       'popular_genre'],
      dtype='str')

In [10]:
def parse_genres(genre_str: str) -> list[str]:
    """
    Parse a genre string into a list of cleaned genre names.
    Handles quoted comma-separated values like '"Blues,Jazz,Vocal"'
    """
    if pd.isna(genre_str):
        return []
    return [g.strip() for g in genre_str.strip('"').split(",")]

In [11]:
# assign most common genres to each audio
all_genres = []
genre_lists = relevant_meta["genres"].apply(parse_genres)

for genres in genre_lists:
    all_genres.extend(genres)
    
from collections import Counter
genre_frequencies = Counter(all_genres)

In [12]:
genre_frequencies

Counter({'Pop/Rock': 881,
         'Electronic': 213,
         'R&B': 213,
         'Country': 179,
         'Jazz': 136,
         'Vocal': 133,
         'Rap': 126,
         'International': 106,
         'Blues': 104,
         'Holiday': 51,
         'Folk': 49,
         'Religious': 39,
         'Reggae': 37,
         'Stage & Screen': 36,
         'Latin': 24,
         'Contemporary Pop/Rock': 21,
         'New Age': 20,
         'Alternative/Indie Rock': 20,
         'Easy Listening': 19,
         'Heavy Metal': 17,
         'Adult Alternative Pop/Rock': 11,
         'Rock & Roll': 10,
         'Club/Dance': 9,
         'Hard Rock': 9,
         'Comedy/Spoken': 9,
         'Album Rock': 8,
         'Alternative Pop/Rock': 8,
         'Hardcore Rap': 8,
         "Children's": 7,
         'Adult Contemporary': 7,
         'Speed/Thrash Metal': 7,
         'Classical': 6,
         'Early Pop/Rock': 6,
         'Gangsta Rap': 6,
         'Golden Age': 6,
         'Dance-Pop': 5,
     

In [13]:
# for each file, pick the genre with the highest global frequency
def pick_most_frequent_genre(genre_str):
    genres = parse_genres(genre_str=genre_str)
    if not genres:
        return None
    return max(genres, key=lambda g:genre_frequencies[g])

In [14]:
relevant_meta["popular_genre"] = relevant_meta["genres"].apply(pick_most_frequent_genre)

In [15]:
relevant_meta.head()

,audio_file_stem,lyric_file_stem,quadrant,genres,popular_genre
0,MT0000008995,MT0000008995,Q3,"Blues,Jazz,Vocal",Jazz
1,MT0000011357,MT0000011357,Q2,"Electronic,International,Pop/Rock",Pop/Rock
2,MT0000026571,MT0000026571,Q4,Vocal,Vocal
3,MT0000027422,MT0000027422,Q3,Pop/Rock,Pop/Rock
4,MT0000031210,MT0000031210,Q3,"Blues,Jazz,Vocal",Jazz


In [16]:
Counter(relevant_meta["popular_genre"])

Counter({'Pop/Rock': 881,
         'Country': 125,
         'Jazz': 113,
         'R&B': 112,
         'Rap': 105,
         'Electronic': 44,
         'Vocal': 38,
         'Blues': 35,
         'Reggae': 30,
         'Religious': 11,
         'International': 9,
         'Holiday': 8,
         'Folk': 7,
         'Stage & Screen': 3,
         'Latin': 3,
         'New Age': 2,
         'Alternative/Indie Rock': 2,
         'Heavy Metal': 2,
         "Children's": 1})

### ^^ HUGE CLASS IMBALANCE - HOW WILL THAT IMPACT CLUSTERING??

In [17]:
relevant_meta = relevant_meta.dropna(subset=["popular_genre"])

In [18]:
relevant_meta.info()

<class 'pandas.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   audio_file_stem  1531 non-null   str  
 1   lyric_file_stem  1531 non-null   str  
 2   quadrant         1531 non-null   str  
 3   genres           1531 non-null   str  
 4   popular_genre    1531 non-null   str  
dtypes: str(5)
memory usage: 59.9 KB


In [19]:
min_frequency = 10

In [20]:
# Filter rare genres
genre_counts = relevant_meta["popular_genre"].value_counts()
valid_genres = genre_counts[genre_counts >= min_frequency].index
popular_genres = relevant_meta[relevant_meta["popular_genre"].isin(valid_genres)].copy()
print(f"After filtering rare genres (<{min_frequency}): {len(relevant_meta)}")
print(f"Genres remaining : {sorted(relevant_meta['popular_genre'].unique())}")

After filtering rare genres (<10): 1531
Genres remaining : ['Alternative/Indie Rock', 'Blues', "Children's", 'Country', 'Electronic', 'Folk', 'Heavy Metal', 'Holiday', 'International', 'Jazz', 'Latin', 'New Age', 'Pop/Rock', 'R&B', 'Rap', 'Reggae', 'Religious', 'Stage & Screen', 'Vocal']


In [21]:
m = ~relevant_meta['audio_file_stem'].isin(popular_genres['audio_file_stem'])

In [22]:
removed_audio_files = relevant_meta[m]["audio_file_stem"]
# keeping track in case we want to add these rare genres later on? are we keeping rare genres now?

In [23]:
removed_audio_files

24      MT0000128573
53      MT0000277032
70      MT0000335310
71      MT0000348553
265     MT0001671399
352     MT0002477987
385     MT0002859848
443     MT0003402178
450     MT0003473344
557     MT0004550458
675         A067-104
685     MT0006169551
701     MT0006431184
718     MT0006633147
730     MT0006747288
754     MT0006983241
766     MT0007087238
867     MT0008428932
957     MT0009741521
982     MT0010203716
984     MT0010261584
1026    MT0010687397
1113    MT0011937155
1117    MT0011944405
1136    MT0012017035
1147    MT0012330276
1203    MT0013583287
1263    MT0015026293
1325    MT0026811443
1351    MT0028335228
1370    MT0028758101
1380            A032
1423    MT0031868141
1454    MT0033218453
1458    MT0033376569
1514    MT0041030749
1525            A171
Name: audio_file_stem, dtype: str

In [24]:
removed_audio_files.shape

(37,)

In [25]:
popular_genres.shape

(1494, 5)

In [26]:
def build_label_map(labels: pd.Series) -> dict[str, int]:
    """Map genre strings to integer indices, sorted by frequency (most common = 0)."""
    freq = Counter(labels.dropna())
    sorted_genres = [genre for genre, _ in freq.most_common()]
    return {genre: idx for idx, genre in enumerate(sorted_genres)}

In [27]:
label_map = build_label_map(popular_genres["popular_genre"])
popular_genres["label"] = popular_genres["popular_genre"].map(label_map)
label_map = build_label_map(relevant_meta["popular_genre"])
relevant_meta["label"] = relevant_meta["popular_genre"].map(label_map)

In [28]:
popular_genres_metadata_file = Path("../../../data/metadata/metadata_popular_en.csv")

In [29]:
popular_genres.to_csv(popular_genres_metadata_file, index=False)

In [30]:
relevant_meta

,audio_file_stem,lyric_file_stem,quadrant,genres,popular_genre,label
0,MT0000008995,MT0000008995,Q3,"Blues,Jazz,Vocal",Jazz,2
1,MT0000011357,MT0000011357,Q2,"Electronic,International,Pop/Rock",Pop/Rock,0
2,MT0000026571,MT0000026571,Q4,Vocal,Vocal,6
3,MT0000027422,MT0000027422,Q3,Pop/Rock,Pop/Rock,0
4,MT0000031210,MT0000031210,Q3,"Blues,Jazz,Vocal",Jazz,2
...,...,...,...,...,...,...
1526,A095-113,L145-113,Q1,R&B,R&B,3
1527,A088-136,L138-136,Q4,"Traditional Pop,Standards,Vocal,Vocal Jazz,Jazz",Jazz,2
1528,A066-140,L116-140,Q4,"Holiday,Holidays,Christmas,Celtic New Age,Ethn...",Pop/Rock,0
1529,A137-153,L027-153,Q4,"Jazz Blues,Vocal Jazz,Jazz,Alternative/Indie R...",Pop/Rock,0


In [31]:
relevant_meta.to_csv(METADATA_FILE, index=False)

In [32]:
removed_audio_file = Path("../../../data/metadata/removed_stems.txt")
with open(removed_audio_file, "w") as f:
    f.write("\n".join(removed_audio_files))

In [33]:
len(Counter(relevant_meta["popular_genre"]))

19